# Intermediate 10 — Sufficiency and Exponential Families

Practice notebook for the **"Sufficiency and Exponential Families"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


## Part 1 — What is a sufficient statistic?

The PDF defines a **sufficient statistic** $T(X_1,\dots,X_n)$ for a parameter $\theta$ as
one for which the conditional distribution of the data given $T$ does not depend on $\theta$:

$$
f(x_1,\dots,x_n \mid T=t;\,\theta) \;\text{ is free of }\theta.
$$

Intuitively, once you know $T$, the rest of the sample carries no additional information
about $\theta$.

For a normal sample $X_1,\dots,X_n \stackrel{iid}{\sim} \mathcal{N}(\mu,\sigma^2)$ with
$\sigma^2$ known, the PDF shows $\bar X$ (equivalently $\sum X_i$) is sufficient for $\mu$.
We verify this **empirically**: conditional on $\bar X = \bar x$, the distribution of the
*centered residuals* $X_i - \bar X$ should not depend on $\mu$. We Monte-Carlo this by
drawing many samples with two very different $\mu$'s, conditioning on $\bar X \approx$
a fixed value, and checking the residuals' distribution is the same.


In [ ]:
# Sufficiency of X-bar for mu (sigma^2 known): the conditional distribution of
# X_i - X-bar given X-bar does not depend on mu. We verify by rejection sampling:
# keep only samples whose X-bar is close to a target t, then compare residuals.

sigma = 1.5
n = 8
target_bar = 5.0   # we condition on X-bar == target_bar (within a tolerance)
tol = 0.05

rng = np.random.default_rng(7)

def residual_dist(mu, N_keep=4000, N_batch=200_000):
    kept = []
    while len(kept) < N_keep:
        X = rng.normal(mu, sigma, size=(N_batch, n))
        xb = X.mean(axis=1)
        mask = np.abs(xb - target_bar) < tol
        kept.extend(X[mask].tolist())
        if len(kept) > 2 * N_keep:  # safety break
            break
    kept = np.array(kept[:N_keep])
    return kept - kept.mean(axis=1, keepdims=True)  # residuals X_i - X-bar

R_low  = residual_dist(mu=0.0)   # mu far from target
R_high = residual_dist(mu=50.0)  # mu very far from target

# Compare the distribution of the first coordinate of the residual vector
print("Residual X1 - Xbar | Xbar~=5, mu=0 :", f"mean={R_low[:,0].mean():.4f}, std={R_low[:,0].std():.4f}")
print("Residual X1 - Xbar | Xbar~=5, mu=50:", f"mean={R_high[:,0].mean():.4f}, std={R_high[:,0].std():.4f}")
print()
# Two-sample KS test: do the residual distributions differ?
ks = stats.ks_2samp(R_low[:,0], R_high[:,0])
print(f"KS statistic = {ks.statistic:.4f}, p-value = {ks.pvalue:.4f}")
print("-> large p-value means the two residual distributions are indistinguishable")

# Plot overlaid histograms
plt.hist(R_low[:,0], bins=40, alpha=0.5, density=True, label=r"$\mu=0$")
plt.hist(R_high[:,0], bins=40, alpha=0.5, density=True, label=r"$\mu=50$")
plt.axvline(0, color="k", linestyle="--", alpha=0.6)
plt.xlabel(r"$X_1 - \bar X$ (residual)")
plt.ylabel("density")
plt.title(r"Residuals given $\bar X \approx 5$ do not depend on $\mu$")
plt.legend()
plt.tight_layout()
plt.show()

# Theory: Var(X_i - X-bar) = sigma^2 (1 - 1/n)
print(f"theory std of residual = {sigma*np.sqrt(1 - 1/n):.4f}")


## Part 2 — Factorization theorem (Neyman–Fisher)

The PDF's **factorization theorem** states that $T(\mathbf{X})$ is sufficient for $\theta$
if and only if the joint density/pmf factors as

$$
f(\mathbf{x};\theta) \;=\; g_\theta\big(T(\mathbf{x})\big)\, h(\mathbf{x}),
$$

with $h$ not depending on $\theta$. Below we apply this to three families, identifying
the sufficient statistic by reading off the factor that contains $\theta$.

| Family | Parameter | Sufficient statistic $T$ |
|---|---|---|
| Bernoulli($p$) | $p$ | $\sum X_i$ (number of successes) |
| Poisson($\lambda$) | $\lambda$ | $\sum X_i$ |
| Normal($\mu,\sigma^2$), $\sigma^2$ known | $\mu$ | $\sum X_i$ (or $\bar X$) |

We now **numerically verify the factorization** for the Poisson case: the joint pmf
$\prod_i \frac{\lambda^{x_i} e^{-\lambda}}{x_i!} = \lambda^{\sum x_i} e^{-n\lambda}\cdot \prod_i \frac{1}{x_i!}$
splits into $g_\lambda(T=\sum x_i) = \lambda^T e^{-n\lambda}$ times $h(\mathbf{x})=\prod_i 1/x_i!$.


In [ ]:
# Verify the Poisson factorization numerically.
# Joint pmf: f(x;lam) = lam^S * exp(-n*lam) * prod(1/x_i!)
# Factor:   g_lam(S)  * h(x) where h(x) = prod(1/x_i!)

lam = 2.3
n = 6
rng = np.random.default_rng(123)
x = rng.poisson(lam, size=n)
S = x.sum()

# Compute joint pmf two ways:
# 1) directly as product of Poisson pmfs
joint_direct = np.prod(stats.poisson.pmf(x, lam))

# 2) via factorization g_lam(S) * h(x)
g = lam**S * np.exp(-n * lam)
h = np.prod(1.0 / np.array([np.math.factorial(int(xi)) for xi in x]))
joint_factored = g * h

print("x                =", x)
print("S = sum(x)       =", S)
print("joint pmf direct =", joint_direct)
print("joint pmf via    = g_lam(S) * h(x) =", joint_factored)
print("abs diff         =", abs(joint_direct - joint_factored))
print()
# h(x) does NOT depend on lam: changing lam only changes g.
for lam2 in [0.7, 5.0, 13.0]:
    g2 = lam2**S * np.exp(-n * lam2)
    print(f"  lam={lam2:5.1f} -> g_lam(S)={g2:.6e}, joint={g2*h:.6e}, "
          f"h(x) unchanged={h:.6e}")


## Part 3 — Exponential families and natural statistics

The PDF defines a family of densities to be an **exponential family** when it can be
written as

$$
f(x;\theta) \;=\; h(x)\,\exp\big(\eta(\theta)^\top T(x) - A(\theta)\big).
$$

Here $\eta(\theta)$ is the **natural parameter**, $T(x)$ the **natural statistic**,
and $A(\theta)$ the **log-partition function**. A fundamental identity (the
moment-generating property of $A$) is

$$
E_\theta[T(X)] \;=\; \nabla_\eta A(\theta).
$$

Below we work with the canonical one-parameter exponential family
$T(x) = x$, $\eta = \theta$, $A(\theta) = \log(1 - e^\theta)$, $h(x)=1$, which is the
$\text{Geometric}(p)$ family with $\theta = \log(1-p)$. We verify
$E[T] = \frac{e^\theta}{1 - e^\theta} = A'(\theta)$ numerically — by sampling and by
finite-difference of the log-partition.


In [ ]:
# One-parameter exponential family: Geometric(p) on {1,2,3,...}
# pmf: P(X=x) = (1-p)^{x-1} * p. Reparametrize theta = log(1-p) (theta < 0).
# Then P(X=x) = exp(theta*(x-1)) * (1 - exp(theta)), so:
#   T(x) = x-1,  eta = theta,  A(theta) = -log(1 - exp(theta)),  h(x)=1.

def log_partition(theta):
    # A(theta) = -log(1 - exp(theta))
    return -np.log1p(-np.exp(theta))

def dA_numeric(theta, eps=1e-6):
    return (log_partition(theta + eps) - log_partition(theta - eps)) / (2 * eps)

rng = np.random.default_rng(2024)

for p in [0.2, 0.5, 0.8]:
    theta = np.log(1 - p)
    n_sim = 200_000
    X = rng.geometric(p, size=n_sim)
    T_emp = X - 1
    E_T_emp = T_emp.mean()
    E_T_theory = np.exp(theta) / (1 - np.exp(theta))   # A'(theta)
    E_T_fd = dA_numeric(theta)
    print(f"p={p:.2f}, theta={theta:.4f}")
    print(f"   E[T] theory (A')   = {E_T_theory:.4f}")
    print(f"   E[T] finite-diff   = {E_T_fd:.4f}")
    print(f"   E[T] simulation   = {E_T_emp:.4f}")
    print()


## Part 4 — Identifying sufficient statistics for Bernoulli and Normal

We apply the factorization theorem directly:

- **Bernoulli($p$)**: $f(\mathbf{x};p) = \prod_i p^{x_i}(1-p)^{1-x_i} = p^{S}(1-p)^{n-S}$
  with $S=\sum X_i$. Here $g_p(S) = p^S(1-p)^{n-S}$ and $h(\mathbf{x})=1$, so
  $S=\sum X_i$ is sufficient for $p$.
- **Normal($\mu,\sigma^2$) with both unknown**: expanding $\sum (x_i-\mu)^2$ one finds
  the joint density depends on the data through $\sum X_i$ **and** $\sum X_i^2$, so the
  pair $\big(\sum X_i, \sum X_i^2\big)$ is sufficient for $(\mu,\sigma^2)$.

We verify the Bernoulli case: the conditional distribution of $\mathbf{X}$ given
$S = \sum X_i = s$ should be **uniform over all $\binom{n}{s}$ arrangements** of $s$
ones — i.e. it does not depend on $p$.


In [ ]:
# Bernoulli sufficiency: conditional on S = number of successes, the data's
# distribution is uniform over all binary strings with exactly s ones, independent of p.
n = 6
s_target = 3
N = 400_000

rng = np.random.default_rng(99)

for p in [0.1, 0.5, 0.9]:
    # Generate many Bernoulli samples, keep those with exactly s_target successes.
    draws = []
    while len(draws) < 20_000:
        batch = rng.binomial(1, p, size=(N, n))
        ok = batch.sum(axis=1) == s_target
        draws.extend(batch[ok].tolist())
    draws = np.array(draws[:20_000])  # each row is a binary string with s_target ones

    # Check: empirical distribution over each pattern should be ~ uniform over
    # the C(n, s_target) possible patterns, regardless of p.
    # Encode each row as a binary integer for histogramming.
    keys = (draws * (2 ** np.arange(n - 1, -1, -1))).sum(axis=1)
    uniq, counts = np.unique(keys, return_counts=True)
    probs = counts / counts.sum()
    # There are C(n, s_target) valid patterns; uniform prob = 1 / C(n, s_target)
    n_patterns = int(np.math.comb(n, s_target))
    print(f"p={p:.1f}: observed {len(uniq)} patterns, "
          f"mean prob={probs.mean():.5f}, uniform={1/n_patterns:.5f}, "
          f"max|prob-1/C|={np.max(np.abs(probs - 1/n_patterns)):.4f}")

# Visualize one case as a bar chart
p = 0.3
draws = []
while len(draws) < 20_000:
    batch = rng.binomial(1, p, size=(N, n))
    ok = batch.sum(axis=1) == s_target
    draws.extend(batch[ok].tolist())
draws = np.array(draws[:20_000])
keys = (draws * (2 ** np.arange(n - 1, -1, -1))).sum(axis=1)
uniq, counts = np.unique(keys, return_counts=True)
plt.bar(uniq.astype(str), counts / counts.sum())
plt.axhline(1 / n_patterns, color="r", linestyle="--", label=r"uniform $1/\binom{n}{s}$")
plt.xlabel("binary pattern (as integer)")
plt.ylabel(r"$P(\mathbf{X} \mid S=s)$")
plt.title(f"Conditional on $S={s_target}$ does not depend on $p$")
plt.legend()
plt.tight_layout()
plt.show()


## Part 5 — Sufficient statistic as a data compressor

The factorization theorem says the sufficient statistic $T$ **summarizes** everything in
the sample that is informative about $\theta$. Operationally: two different samples with
the same value of $T$ lead to the same likelihood (up to a $\theta$-free factor).

Below we draw several normal samples, all of which happen to share the same sample sum
$S=\sum X_i$ (we round), and confirm the **likelihoods as functions of $\mu$** are
identical — only $h(\mathbf{x})$ differs. This is the factorization theorem in action.

**Your turn:** change $\sigma$, $n$, and the target sum; re-run and confirm the
likelihood curves still coincide when $S$ matches. Then try conditioning on the sample
mean of the absolute values instead of $\sum X_i$ — the curves should no longer match
(because that statistic is *not* sufficient for $\mu$).


In [ ]:
# Two different samples sharing the SAME sample sum S should give the same
# likelihood L(mu) up to a mu-independent constant.

sigma = 2.0
n = 5

rng = np.random.default_rng(314)

# Construct two samples with the same sum (target S0)
S0 = 12.0
while True:
    a = rng.normal(0, sigma, size=n)
    if abs(a.sum() - S0) < 1e-8:
        break
while True:
    b = rng.normal(0, sigma, size=n)
    if abs(b.sum() - S0) < 1e-8 and not np.allclose(np.sort(a), np.sort(b)):
        break

# Force exact equality of the sums by shifting (negligible adjustment)
a = a + (S0 - a.sum()) / n
b = b + (S0 - b.sum()) / n
print("sample A:", np.round(a, 4), " sum =", a.sum())
print("sample B:", np.round(b, 4), " sum =", b.sum())

mu_grid = np.linspace(-3, 9, 400)
def loglik(x, mu):
    return np.sum(stats.norm.logpdf(x, loc=mu, scale=sigma))

LL_a = np.array([loglik(a, m) for m in mu_grid])
LL_b = np.array([loglik(b, m) for m in mu_grid])

# The log-likelihoods differ by a constant (the log h(x) term).
diff = LL_a - LL_b
print()
print("LL_A(mu) - LL_B(mu): mean =", round(diff.mean(), 6),
      " std =", round(diff.std(), 12))
print("-> constant offset => likelihood curves identical up to mu-free factor")

plt.plot(mu_grid, LL_a, label="sample A")
plt.plot(mu_grid, LL_b, "--", label=r"sample B (same $S=\sum X_i$)")
plt.xlabel(r"$\mu$")
plt.ylabel(r"$\log L(\mu)$")
plt.title(r"Same $\sum X_i \Rightarrow$ same likelihood shape (factorization theorem)")
plt.legend()
plt.tight_layout()
plt.show()


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Use the factorization theorem to identify a sufficient statistic for $\lambda$ in a sample $X_1,\dots,X_n\stackrel{iid}{\sim}\text{Poisson}(\lambda)$. Justify by writing the factorization $g_\lambda(T)\,h(\mathbf{x})$.

2. For $X_1,\dots,X_n\stackrel{iid}{\sim}\mathcal{N}(\mu,\sigma^2)$ with **both** parameters unknown, show by expanding $\sum (x_i-\mu)^2$ that the pair $\big(\sum X_i,\,\sum X_i^2\big)$ is sufficient for $(\mu,\sigma^2)$.

3. Write the Bernoulli($p$) pmf in exponential-family form and identify $\eta$, $T(x)$, and $A(\eta)$. Then verify $E[T]=A'(\eta)$ numerically with a small simulation at $p=0.3$.

4. In Part 1, why did we condition on $\bar X \approx 5$ (a tolerance window) rather than exactly $\bar X = 5$? What would go wrong with exact conditioning in a continuous model?

5. Give the natural statistic and log-partition for the exponential distribution $f(x;\lambda)=\lambda e^{-\lambda x}$, $x>0$, and confirm $E[T]=A'(\eta)$ at $\lambda=2$ by simulation.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** $f(\mathbf{x};\lambda)=\prod_i \frac{\lambda^{x_i}e^{-\lambda}}{x_i!}=\lambda^{\sum x_i}e^{-n\lambda}\cdot\prod_i \frac{1}{x_i!}$. So $g_\lambda(T)=\lambda^T e^{-n\lambda}$ with $T=\sum X_i$ and $h(\mathbf{x})=\prod_i 1/x_i!$, which is $\theta$-free. Hence $\sum X_i$ is sufficient for $\lambda$.

**2.** $\sum (x_i-\mu)^2=\sum x_i^2 - 2\mu\sum x_i + n\mu^2$, so
$$
f(\mathbf{x};\mu,\sigma^2)\propto \sigma^{-n}\exp\!\Big(\!-\tfrac{1}{2\sigma^2}\big[\sum x_i^2 - 2\mu\sum x_i + n\mu^2\big]\Big).
$$
The $\sigma$-depending part is $\sigma^{-n}\exp\!\big(\!-\tfrac{1}{2\sigma^2}[\sum x_i^2 - 2\mu\sum x_i + n\mu^2]\big)$, which depends on $\mathbf{x}$ only through $\sum x_i$ and $\sum x_i^2$. The $\theta$-free remainder $h(\mathbf{x})=1$. Thus $\big(\sum X_i,\sum X_i^2\big)$ is sufficient for $(\mu,\sigma^2)$.

**3.** $p^{x}(1-p)^{1-x}=\exp\!\big(\log\tfrac{p}{1-p}\cdot x + \log(1-p)\big)$, so $\eta=\log\tfrac{p}{1-p}$, $T(x)=x$, $A(\eta)=-\log(1-p)=\log(1+e^\eta)$. Then $A'(\eta)=\frac{e^\eta}{1+e^\eta}=p=E[X]$. At $p=0.3$, $A'(\eta)=0.3$; a simulation `rng.binomial(1,0.3,size=200_000).mean()` gives $\approx 0.300$.

**4.** In a continuous model $P(\bar X = t)=0$, so exact conditioning is undefined (the conditional distribution degenerates). Conditioning on a small window $\bar X\in[t\pm\varepsilon]$ gives a well-defined conditional distribution that converges to the exact one as $\varepsilon\to0$; this is the standard Monte-Carlo approach to continuous conditional distributions.

**5.** Write $f(x;\lambda)=\lambda e^{-\lambda x}=\exp(-\lambda x + \log\lambda)$, so $\eta=-\lambda$, $T(x)=x$, $A(\eta)=-\log(-\eta)$ (since $\lambda=-\eta>0$). Then $A'(\eta)=-1/\eta=1/\lambda=E[X]$. At $\lambda=2$, $\eta=-2$, $A'(-2)=1/2$; simulation `rng.exponential(scale=1/2,size=200_000).mean()` gives $\approx 0.500$.

</details>
